# Lab 2 -  Feature Selection


In [1]:
import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.12.3
NumPy   : 2.4.3
Seed    : 414


In [ ]:

import os
import fastf1
import pandas as pd
import numpy as np

# ── 1. Cache ──────────────────────────────────────────────────────────────────
cache_path = os.path.abspath(os.path.join(os.getcwd(), "data", "fastf1_cache"))
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

# ── 2. Configuración ──────────────────────────────────────────────────────────
RACES = {
    2022: ["Bahrain", "Monaco", "Abu Dhabi"],
    2023: ["Bahrain", "Monaco", "Abu Dhabi"],
}

# ── 3. Helpers ────────────────────────────────────────────────────────────────

def _time_to_seconds(t) -> float:
    """Convierte timedelta / Timedelta de FastF1 a segundos float (NaN si nulo)."""
    try:
        return t.total_seconds()
    except Exception:
        return np.nan


def load_qualifying_features(year: int, track: str) -> pd.DataFrame:
    """
    Devuelve un DataFrame con una fila por piloto y columnas:
        Driver, QualifyingPosition, Q1_s, Q2_s, Q3_s, BestQ_s
    """
    print(f"    [Q] {year} {track} ...", end=" ")
    try:
        session = fastf1.get_session(year, track, "Q")
        session.load(telemetry=False, weather=False, messages=False)

        laps = session.laps.copy()

        rows = []
        for driver in laps["Driver"].unique():
            d = laps[laps["Driver"] == driver]

            def best_segment(seg: str) -> float:
                seg_laps = d[d["Sector1SessionTime"].notna()]  
                return np.nan

            rows.append({"Driver": driver})

        res = session.results[
            ["Abbreviation", "Position", "Q1", "Q2", "Q3"]
        ].copy()
        res = res.rename(columns={
            "Abbreviation": "Driver",
            "Position":     "QualifyingPosition",
        })
        res["Q1_s"]   = res["Q1"].apply(_time_to_seconds)
        res["Q2_s"]   = res["Q2"].apply(_time_to_seconds)
        res["Q3_s"]   = res["Q3"].apply(_time_to_seconds)
        # Mejor tiempo disponible entre Q1/Q2/Q3
        res["BestQ_s"] = res[["Q1_s", "Q2_s", "Q3_s"]].min(axis=1)
        res["QualifyingPosition"] = (
            pd.to_numeric(res["QualifyingPosition"], errors="coerce").astype("Int64")
        )

        res = res[["Driver", "QualifyingPosition", "Q1_s", "Q2_s", "Q3_s", "BestQ_s"]]
        print(f"OK  ({len(res)} pilotos)")
        return res

    except Exception as exc:
        print(f"ERROR — {exc}")
        return pd.DataFrame()


def load_race_results(year: int, track: str) -> pd.DataFrame:
    """
    Devuelve un DataFrame con una fila por piloto y columnas de resultado
    de carrera: posición final, grid, puntos, status, métricas de vueltas.
    """
    print(f"    [R] {year} {track} ...", end=" ")
    try:
        session = fastf1.get_session(year, track, "R")
        session.load(telemetry=False, weather=False, messages=False)

        # -- Resultados base --------------------------------------------------
        res = session.results[[
            "Abbreviation", "GridPosition", "Position",
            "Points", "Status", "TeamName"
        ]].copy()
        res = res.rename(columns={
            "Abbreviation": "Driver",
            "Position":     "FinalPosition",
        })
        for col in ("GridPosition", "FinalPosition"):
            res[col] = pd.to_numeric(res[col], errors="coerce").astype("Int64")
        res["Points"] = pd.to_numeric(res["Points"], errors="coerce")

        # -- Métricas de vueltas por piloto -----------------------------------
        laps = session.laps.copy()
        laps["LapTime_s"] = laps["LapTime"].apply(_time_to_seconds)

        lap_stats = (
            laps.groupby("Driver")["LapTime_s"]
            .agg(
                MedianLapTime_s="median",
                StdLapTime_s="std",
                FastestLap_s="min",
                LapsCompleted="count",
            )
            .reset_index()
        )

        # Número de pit stops
        pit_counts = (
            laps[laps["PitOutTime"].notna()]
            .groupby("Driver")
            .size()
            .reset_index(name="PitStops")
        )

        df = res.merge(lap_stats,  on="Driver", how="left")
        df = df.merge(pit_counts, on="Driver", how="left")
        df["PitStops"] = df["PitStops"].fillna(0).astype(int)

        print(f"OK  ({len(df)} pilotos)")
        return df

    except Exception as exc:
        print(f"ERROR — {exc}")
        return pd.DataFrame()



def build_dataset(races: dict[int, list[str]]) -> pd.DataFrame:
    """
    Para cada (año, carrera):
      - Carga Qualifying  → features de clasificación
      - Carga Race        → target + features históricas de carrera
      - Une ambas por Driver
      - Agrega columnas de contexto (Year, Track)
      - Calcula el target binario IsTop10
    """
    all_rows = []

    for year, track_list in races.items():
        print(f"\n{'='*55}")
        print(f"  {year}")
        print(f"{'='*55}")

        for track in track_list:
            print(f"\n  ── {track} ──")

            df_qual = load_qualifying_features(year, track)
            df_race = load_race_results(year, track)

            if df_qual.empty or df_race.empty:
                print(f"  ⚠  Saltando {year} {track} (datos incompletos)")
                continue

            # Unir qualifying + race por piloto
            df = df_race.merge(df_qual, on="Driver", how="left")

            # Contexto
            df["Year"]  = year
            df["Track"] = track

            df["IsTop10"] = (df["FinalPosition"] <= 10).astype(int)

            all_rows.append(df)

    if not all_rows:
        print("\n⚠  No se pudo construir ningún registro.")
        return pd.DataFrame()

    dataset = pd.concat(all_rows, ignore_index=True)
    return dataset


print("Construyendo dataset...\n")
df_dataset = build_dataset(RACES)

if not df_dataset.empty:
    print(f"\n{'='*55}")
    print(f"Dataset final: {df_dataset.shape[0]} filas × {df_dataset.shape[1]} columnas")
    print(f"{'='*55}")

    print("\nDistribución del target IsTop10:")
    print(df_dataset["IsTop10"].value_counts().rename({1: "Top 10", 0: "Fuera Top 10"}))

    print("\nBalance por año:")
    print(
        df_dataset.groupby(["Year", "IsTop10"])
        .size()
        .unstack(fill_value=0)
        .rename(columns={0: "Fuera Top 10", 1: "Top 10"})
        .to_string()
    )

    print("\nColumnas del dataset:")
    print(list(df_dataset.columns))

    print("\nNulos por columna (%):")
    null_pct = (df_dataset.isnull().mean() * 100).round(1)
    print(null_pct[null_pct > 0].to_string())

    print("\nMuestra (5 filas):")
    preview_cols = [
        "Year", "Track", "Driver", "TeamName",
        "GridPosition", "QualifyingPosition", "BestQ_s",
        "FinalPosition", "Points", "PitStops",
        "MedianLapTime_s", "FastestLap_s", "LapsCompleted",
        "IsTop10",
    ]
    print(df_dataset[[c for c in preview_cols if c in df_dataset.columns]].head().to_string(index=False))


Construyendo dataset...


  2022

  ── Bahrain ──
    [Q] 2022 Bahrain ... 

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core         

ERROR — The data you are trying to access has not been loaded yet. See `Session.load`
    [R] 2022 Bahrain ... 

logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
logger      WARNING 	Failed to load total lap count!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
logger      WARNING 	Failed to load timing data!
core        WARNING 	Failed

ERROR — The data you are trying to access has not been loaded yet. See `Session.load`
  ⚠  Saltando 2022 Bahrain (datos incompletos)

  ── Monaco ──
    [Q] 2022 Monaco ... 

...

OK  (20 pilotos)

  ── Abu Dhabi ──
    [Q] 2023 Abu Dhabi ... 

req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to

OK  (20 pilotos)
    [R] 2023 Abu Dhabi ... 

core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '11', '4', '81', '14', '22', '44', '18', '3', '31', '10', '23', '27', '2', '24', '55', '77', '20']


OK  (20 pilotos)

Dataset final: 60 filas × 19 columnas

Distribución del target IsTop10:
IsTop10
Top 10          30
Fuera Top 10    30
Name: count, dtype: int64

Balance por año:
IsTop10  Fuera Top 10  Top 10
Year                         
2023               30      30

Columnas del dataset:
['Driver', 'GridPosition', 'FinalPosition', 'Points', 'Status', 'TeamName', 'MedianLapTime_s', 'StdLapTime_s', 'FastestLap_s', 'LapsCompleted', 'PitStops', 'QualifyingPosition', 'Q1_s', 'Q2_s', 'Q3_s', 'BestQ_s', 'Year', 'Track', 'IsTop10']

Nulos por columna (%):
Q1_s        1.7
Q2_s       26.7
Q3_s       51.7
BestQ_s     1.7

Muestra (5 filas):
 Year   Track Driver        TeamName  GridPosition  QualifyingPosition  BestQ_s  FinalPosition  Points  PitStops  MedianLapTime_s  FastestLap_s  LapsCompleted  IsTop10
 2023 Bahrain    VER Red Bull Racing             1                   1   89.708              1    25.0         2           97.651        96.236             57        1
 2023 Bahrain    PER R

In [3]:
df_dataset.columns

Index(['Driver', 'GridPosition', 'FinalPosition', 'Points', 'Status',
       'TeamName', 'MedianLapTime_s', 'StdLapTime_s', 'FastestLap_s',
       'LapsCompleted', 'PitStops', 'QualifyingPosition', 'Q1_s', 'Q2_s',
       'Q3_s', 'BestQ_s', 'Year', 'Track', 'IsTop10'],
      dtype='object')